# Vision Mamba: A Structured State Space Model for Vision

Vision Mamba is a neural architecture designed to combine the global modeling capability of Transformers with the efficiency and recurrence of structured state space models (SSMs). It is part of the growing family of Mamba models, which leverage selective, long-sequence modeling using SSM-based recurrence instead of expensive full attention.

---

## Architecture Overview

Vision Mamba is structured as a **stack of VSS (Vision SSM) blocks**, each composed of:

1. **Patch Embedding Layer**  
   Converts an image into a sequence of embedded patches — similar to ViT (Vision Transformer).

2. **VSS Block (Vision State Space Block)**  
   Each block consists of:
   - A **Linear Projection**
   - A **Depthwise Convolution** (for local mixing)
   - A **SSM Layer** for sequence modeling
   - A **Gating Function**
   - **Residual Connections** and **Layer Normalization**

3. **Classification Head**  
   A simple MLP or linear classifier that maps the final sequence embedding to class logits.

---

## Core Idea: State Space Modeling (SSM)

The core innovation lies in the **SSM module**, which replaces traditional attention mechanisms.
  
This allows the model to efficiently capture **long-range dependencies** in sequences using:

$$
h_k = A_k \cdot h_{k-1} + B_k \cdot x_{k} \quad \text{with} \quad y_k = C(h_k)
$$

---

## Comparison with Transformers

| Feature                | Transformers     | Vision Mamba         |
|------------------------|------------------|-----------------------|
| Token Mixing           | Full Attention   | SSM-based Recurrence  |
| Inductive Bias         | None             | Recurrent + Locality  |
| Computational Cost     | Quadratic (O(L²))| Linear (O(L))         |
| Long-Range Modeling    | Explicit         | Implicit via SSM      |


## Model Components Breakdown

- **PatchEmbed**: Converts images to patch sequences
- **SSM Module**: Performs the core recurrence modeling
- **VSSBlock**: Encapsulates local mixing + SSM + residual logic
- **MLP Head**: Outputs class logits from global features

---

## References And Orginal papers used in this notebook

- [Mamba: Linear-Time Sequence Modeling with Selective SSMs](https://arxiv.org/abs/2312.00752)
- [Vision Mamba: Efficient Visual Representation Learning with Bidirectional State Space Model](https://arxiv.org/abs/2401.09417)

## Import libraries

In [1]:
import time

import torch
import torch.nn as nn

import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

import torch.optim as optim

from torch.jit import script

### Architecture

In this notebook we will implement the vision mamba with respect to architecture given in orginal paper

![vim_architecture](./screenshots/Vim_architecture.jpg)

## Patch Embedding and Positional Embedding

In Vision Mamba, the input image is divided into smaller patches (e.g., 4×4), and each patch is flattened and projected into a high-dimensional embedding space using a linear layer. This forms the patch embeddings, which convert spatial image data into a sequence format.

To help the model understand spatial order, positional embeddings are added to each patch embedding. These embeddings inject information about the position of each patch in the sequence, which is crucial since the model processes the sequence without inherent spatial awareness.

This process mimics the approach used in Vision Transformers (ViT) and ensures that both content and position are available for downstream processing.

We add CLS token in PositionalEmbedding too.

In [2]:
class PatchEmbedding(nn.Module):
    def __init__(self, image_size, patch_size, in_channels, embed_dim):
        super().__init__()
        self.image_size = image_size
        self.patch_size = patch_size
        self.num_patches = (image_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, 
                              stride=patch_size)
    
    def forward(self, x):
        # x.shape: (batch_size, in_channels, image_size, image_size)
        x = self.proj(x) # (batch_size, embed_dim, H/patch_size, W/patch_size)
        x = x.flatten(2) # (batch_size, embed_dim, num_patches)
        x = x.transpose(1, 2) # (batch_size, num_patches, embed_dim)
        return x
    
class PositionalEmbedding(nn.Module):
    def __init__(self, num_patches, embed_dim):
        super().__init__()
        self.cls_token = nn.Parameter(torch.randn(1, embed_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, embed_dim))

    def forward(self, x):
        # x.shape: (batch_size, num_patches, embed_dim)
        cls_token = self.cls_token.expand(x.size(0), -1, -1)
        x = torch.cat([cls_token, x], dim=1) # (batch_size, num_patches + 1, embed_dim)
        x = x + self.pos_embed
        return x

## Depthwise Convolution in Vision Mamba

In Vision Mamba, depthwise convolution is applied to enhance local feature extraction before the Mamba block processes the sequence globally. 

Unlike standard convolution that mixes information across channels, depthwise convolution performs separate convolutions for each input channel. This is done by setting the `groups` parameter in PyTorch’s `Conv1d` equal to `embed_dim`. It ensures that each channel is convolved independently, reducing computation while preserving per-channel patterns.

This step helps the model focus on short-range dependencies in the sequence while the Mamba block handles long-range interactions.

In [3]:
class DepthWiseConv(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()

        self.conv_forward = nn.Conv1d(embed_dim, embed_dim, kernel_size=3, 
                                      padding=1, groups=embed_dim)
        self.conv_backward = nn.Conv1d(embed_dim, embed_dim, kernel_size=3,
                                       padding=1, groups=embed_dim)

    def forward(self, x):
        x_reversed = torch.flip(x, dims=[1])
        x = x.transpose(1, 2)
        x_reversed = x_reversed.transpose(1, 2)

        x = self.conv_forward(x)
        x_reversed = self.conv_backward(x)

        x = x.transpose(1, 2)
        x_reversed = torch.flip(x_reversed.transpose(1, 2), dims=[1])

        return x, x_reversed

## State Space Model (SSM) Block in Vision Mamba

This module implements the core of the State Space Model (SSM) used in Vision Mamba to model long-range dependencies in token sequences.

### Overview

Given a sequence of token embeddings $x \in \mathbb{R}^{B \times M \times D}$, where $B$ is the batch size, $M$ is the sequence length, and $D$ is the embedding dimension, the SSM performs a **discretized recurrence** over time using dynamically modulated transition matrices. The SSM is inspired by the Mamba architecture and adapted for vision tasks to efficiently capture spatial and long-range dependencies in image token sequences.

### Mathematical Details

The SSM is based on a continuous-time linear time-invariant (LTI) system, which is discretized for processing discrete token sequences. The core equations and their mathematical underpinnings are described below.

#### Continuous-Time SSM
The SSM models a hidden state $h(t) \in \mathbb{R}^{N}$ (where $N$ is the state dimension) evolving over time according to the following differential equations:
$$
h'(t) = \mathbf{A}h(t) + \mathbf{B}u(t)
$$
$$
y(t) = \mathbf{C}h(t) + \mathbf{D}u(t)
$$
Where:
- $h(t)$: Hidden state at time $t$.
- $u(t) \in \mathbb{R}$: Input scalar (a single dimension of the token embedding).
- $y(t) \in \mathbb{R}$: Output scalar.
- $\mathbf{A} \in \mathbb{R}^{N \times N}$: State transition matrix, controlling the dynamics of the hidden state.
- $\mathbf{B} \in \mathbb{R}^{N \times 1}$: Input projection matrix.
- $\mathbf{C} \in \mathbb{R}^{1 \times N}$: Output projection matrix.
- $\mathbf{D} \in \mathbb{R}$: Feedthrough matrix (often set to zero for simplicity).

These equations describe a linear system where the hidden state evolves based on the input and previous state, and the output is a linear combination of the hidden state and input.

#### Discretization
To apply the SSM to discrete token sequences, the continuous-time system is discretized using a zero-order hold (ZOH) assumption. For a time step $\Delta$, the discretized system is:
$$
h_t = \bar{\mathbf{A}} h_{t-1} + \bar{\mathbf{B}} u_t
$$
$$
y_t = \mathbf{C} h_t + \mathbf{D} u_t
$$
Where:
- $\bar{\mathbf{A}} = e^{\mathbf{A}\Delta}$: Discretized state transition matrix, computed via the matrix exponential.
- $\bar{\mathbf{B}} = (\mathbf{A}^{-1}(e^{\mathbf{A}\Delta} - \mathbf{I}))\mathbf{B}$: Discretized input matrix.
- $\Delta$: A learnable discretization step, controlling the time scale of the system.

The matrix exponential $e^{\mathbf{A}\Delta}$ ensures that the continuous dynamics are approximated over discrete steps. For computational efficiency, $\mathbf{A}$ is often structured (e.g., diagonal or low-rank) to simplify the exponential computation.

#### Selective SSM (S4/Mamba)
Vision Mamba employs a **selective SSM**, where the parameters $\mathbf{A}$, $\mathbf{B}$, $\mathbf{C}$, and $\Delta$ are input-dependent to enhance expressiveness. Specifically:
- $\Delta$: A learnable scalar per channel, computed via a linear projection from the input.
- $\mathbf{B}$, $\mathbf{C}$: Generated dynamically via linear projections from the input $u_t$.
- $\mathbf{A}$: Typically initialized as a diagonal or low-rank matrix to ensure stability and efficiency.

The selective SSM computes:
$$
\bar{\mathbf{A}} = e^{\mathbf{A}\Delta}, \quad \bar{\mathbf{B}} = (\mathbf{A}^{-1}(e^{\mathbf{A}\Delta} - \mathbf{I}))\mathbf{B}
$$
For each token $t$, the recurrence is:
$$
h_t = \bar{\mathbf{A}} h_{t-1} + \bar{\mathbf{B}} u_t
$$
$$
y_t = \mathbf{C} h_t
$$
This recurrence is applied across the sequence, with the hidden state $h_t$ capturing long-range dependencies.

#### Convolution-Based Computation
To handle long sequences efficiently, the SSM can be reformulated as a convolution. The output $y_t$ is computed as a convolution of the input $u_t$ with a structured kernel $K$:
$$
y_t = (K * u)_t
$$
Where the kernel $K$ is:
$$
K = (\mathbf{C} \bar{\mathbf{B}}, \mathbf{C} \bar{\mathbf{A}} \bar{\mathbf{B}}, \mathbf{C} \bar{\mathbf{A}}^2 \bar{\mathbf{B}}, \dots)
$$
This kernel is computed in the frequency domain using fast Fourier transforms (FFTs), reducing the computational complexity from $O(MN^2)$ (naive recurrence) to $O(M \log M)$, where $M$ is the sequence length.

#### Vision Mamba Adaptations
In Vision Mamba, the SSM is tailored for 2D image token sequences:
- **Bidirectional Processing**: The SSM is applied in multiple directions (e.g., row-wise and column-wise) to capture spatial dependencies in images.
- **Input-Dependent Parameters**: $\mathbf{B}$, $\mathbf{C}$, and $\Delta$ are computed dynamically via linear projections from the input embeddings.
- **Parallel SSMs**: Multiple SSMs may be applied across different dimensions or channels to increase model capacity.

The algorithm and parameters in the original paper are shown in the image below:

![ssm_forward](./screenshots/SSM_forward.jpg)

We will implement this type of SSM (Recurrent SSM) in code cell below.

In [ ]:
class SSM(nn.Module):
    def __init__(self, state_dim, embed_dim):
        super().__init__()
        self.state_dim = state_dim
        self.embed_dim = embed_dim

        self.delta = nn.Linear(embed_dim, embed_dim)
        self.delta_parameter = nn.Parameter(torch.randn(1, 1, embed_dim))

        self.A = nn.Parameter(torch.randn(embed_dim, state_dim)) * 0.1
        self.B = nn.Linear(embed_dim, state_dim)
        self.C = nn.Linear(embed_dim, state_dim)

    def forward(self, x):
        B, M, E = x.shape # (B, M, embed_dim)

        delta_0 = torch.log(1 + torch.exp(self.delta(x) + self.delta_parameter))# (B, M, embed_dim)
        B_0 = self.B(x) # (B, M, state_dim)
        C_0 = self.C(x) # (B, M, state_dim)

        A_bar = delta_0.unsqueeze(-1).to(x.device)  * self.A.unsqueeze(0).unsqueeze(0).to(x.device)  # (B, M, embed_dim, state_dim)
        B_bar = delta_0.unsqueeze(-1).to(x.device)  * B_0.unsqueeze(2).to(x.device)  # (B, M, embed_dim, state_dim)
        A_bar = A_bar.to(x.device)
        B_bar = B_bar.to(x.device)

        y_0 = SSM.ssm_loop(A_bar, B_bar, C_0, x)

        return y_0

    @script # torch script compilation for better performance and exportability.
    def ssm_loop(A_bar, B_bar, C_0, x):
        B, M, E, D = A_bar.shape
        h_0 = torch.zeros(B, E, D, device=x.device)
        y_0 = torch.zeros(B, M, E, device=x.device)
        for i in range(M):
            h_0 = A_bar[:, i, :, :] * h_0 + B_bar[:, i, :, :] * x[:, i, :, None]
            h_0 = h_0.detach()
            y_0[:, i, :] = torch.bmm(h_0, C_0[:, i, :].unsqueeze(-1)).squeeze(-1)
        return y_0

## FastSSM

Traditional State Space Models (SSMs) follow a sequential recurrence of the form:

$
h_t = A h_{t-1} + B x_t, \quad y_t = C h_t
$

While this is mathematically elegant and models temporal dynamics effectively, **it is inherently sequential**, that is:

- **Cannot be parallelized over time**: Each state $h_t$ depends on $h_{t-1}$
- **GPU-inefficient**: GPU hardware is optimized for batched matrix operations, not per-step recurrence
- **Slow inference and training** for long sequences

---

`FastSSM` is a parallel approximation of this recurrence designed to mimic SSM behavior **without sequential state updates**.

---

### Step 1: Dynamic State Parameterization

Instead of fixed matrices $A$, $B$, and $C$, we generate dynamic, elementwise parameters from the input sequence $x_t$:

params = Linear(x) → $[\lambda, u, w]$  
$\lambda$, u, w = params.chunk(3, dim=-1)

These can be interpreted as:

- $\lambda_t$: a learnable state decay coefficient (approximates diagonal $A$)
- $u_t$: input scaling factor (approximates diagonal $B$)
- $w_t$: output gate (replaces $C$)

---

### Step 2: Replacing Recurrence with Convolution

Instead of computing the hidden state recursively as:

$$
h_t = A h_{t-1} + B x_t
$$

We define a modulated input:

$$
\tilde{x}_t = \lambda_t \cdot u_t \cdot x_t
$$

and then compute a causal convolution over time:

$$
h_t = \sum_{k=0}^{K} K_k \cdot \tilde{x}_{t-k}
$$

This can be implemented efficiently using a depthwise `Conv1D`, where the kernel $K_k$ is learned and shared across the batch.

This convolution acts as a **finite impulse response (FIR)** system, which approximates the behavior of the original infinite recurrence.

---

### Step 3: Gated Output Projection

To control the output signal, we use an elementwise gate $w_t$:

$$
y_t = h_t \cdot w_t
$$

This allows the network to learn how much of each filtered response should be kept or suppressed.

---

All of This allows:

- **Parallel processing over time**
- **Fast execution using convolution** (especially depthwise conv)
- **State-space-like behavior** with significantly better runtime performance

### Summary: Why We Use FastSSM

| Feature                  | Recurrent SSM | FastSSM     |
|--------------------------|----------------|------------------|
| Sequential Dependency   | Required     | Removed       |
| Parallel Computation    | Impossible   | Fully Parallel |
| Speed (GPU Utilization) | Poor         | High           |
| Model Expressivity      | Strong       | Approximate but effective |

In [5]:
class FastSSM(nn.Module):
    def __init__(self, embed_dim, kernel_size=7):
        super().__init__()
        self.embed_dim = embed_dim
        self.kernel_size = kernel_size

        self.param_proj = nn.Linear(embed_dim, 3 * embed_dim)
        self.conv = nn.Conv1d(embed_dim, embed_dim, kernel_size, 
                              padding=kernel_size-1, groups=embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        B, M, E = x.shape # (B, M, E)
        params = self.param_proj(x)  # (B, M, 3E)
        lan, u, w = params.chunk(3, dim=-1)

        lan = torch.sigmoid(lan)
        u = torch.tanh(u)
        w = torch.tanh(w)

        x_bar = lan * u * x

        # simulate long-range recurrence with depthwise convolution
        x_bar = x_bar.transpose(1, 2)  # (B, E, M)
        y = self.conv(x_bar)[:, :, :M] # trim extra padding
        y = y.transpose(1, 2) # (B, M, E)

        y = y * w
        return self.out_proj(y)

## Bidirectional State Space Model (BidirectionalSSM)

In many sequence modeling tasks — especially in vision — understanding the context from both past and future is critical. Traditional State Space Models (SSMs) process sequences in a unidirectional manner (typically left-to-right). However, visual patterns often benefit from bidirectional information.

In [6]:
class BidirectionalSSM(nn.Module):
    def __init__(self, state_dim, embed_dim, kernel_size ,ssm_type='recurrent'):
        super().__init__()
        if ssm_type == 'recurrent':
            self.forward_ssm = SSM(state_dim, embed_dim)
            self.reverse_ssm = SSM(state_dim, embed_dim)
        elif ssm_type == 'convolutional':
            self.forward_ssm = FastSSM(embed_dim, kernel_size)
            self.reverse_ssm = FastSSM(embed_dim, kernel_size)
        else:
            raise TypeError('Invalid ssm_type: choose "recurrent" or "convolutional"')

    def forward(self, x, x_reverse):
        y_forward = self.forward_ssm(x)
        
        x_reverse = torch.flip(x_reverse, dims=[1])
        y_reverse = self.reverse_ssm(x_reverse)
        y_reverse = torch.flip(y_reverse, dims=[1])

        return y_forward, y_reverse

### VSSBlock & VimEncoder

Put all of these components together, We have VSS block implemented in original paper.

In [ ]:
class VSSBlock(nn.Module):
    def __init__(self, input_dim, state_dim, embed_dim, kernel_size, ssm_type):
        super().__init__()
        self.norm = nn.LayerNorm(input_dim)

        self.x_linear = nn.Linear(input_dim, embed_dim)
        self.z_linear = nn.Linear(input_dim, embed_dim)

        self.depth_conv = DepthWiseConv(embed_dim)
        self.silu = nn.SiLU()

        self.bi_fastssm = BidirectionalSSM(state_dim, embed_dim, kernel_size, ssm_type)

        self.y_linear = nn.Linear(embed_dim, input_dim)

    def forward(self, x):
        residual = x # skip connection
        x = self.norm(x) # layer norm

        z = self.silu(self.z_linear(x))
        x = self.x_linear(x)
        
        # forward and backward convolutions
        x, x_reversed = self.depth_conv(x)
        x, x_reversed = self.silu(x), self.silu(x_reversed)

        # forward and backward state space model blocks
        x_forward, x_backward = self.bi_fastssm(x, x_reversed)

        # gating
        x_forward = x_forward * z
        x_backward = x_backward * z
        
        y = self.y_linear(x_forward + x_backward) + residual # add skip connection

        return y

Multiple VisionMamba (Vim) blocks.

In [8]:
class VisionMambaEncoder(nn.Module):
    def __init__(self, input_dim, state_dim, embed_dim, depth, kernel_size, ssm_type):
        super().__init__()
        self.blocks = nn.ModuleList([
            VSSBlock(input_dim, embed_dim, state_dim, kernel_size, ssm_type) for _ in range(depth)
        ])

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        return x

### Final Classification Head

This class projects the encoded space to the output class logits. Specifically, after the sequence is processed by the encoder (e.g., VSS blocks), we apply:

1. **Layer Normalization** to stabilize training and smooth the feature distribution.
2. **[CLS] Token Selection**: The first token (prepended during positional embedding) is treated as a summary of the entire sequence.
3. **Linear Projection**: The [CLS] token is passed through a linear layer to map the embedding dimension to the number of classes

The final logits can then be used with a softmax and cross-entropy loss during training.

In [9]:
class ClassificationHead(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.norm = nn.LayerNorm(input_dim)
        self.fc = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        x = self.norm(x)
        x = x[:, 0, :] # select cls token for final classification
        x = self.fc(x)  # (batch_size, num_classes)
        return x

## Full Vim class

In [10]:
class VisionMamba(nn.Module):
    def __init__(
        self,
        image_size=32,
        patch_size=4,
        in_channels=3,
        input_dim=128,
        embed_dim=128,
        state_dim=128,
        kernel_size=7,
        ssm_type='recurrent',
        depth=12,
        num_classes=10
    ):
        super().__init__()
        self.patch_embed = PatchEmbedding(image_size, patch_size, in_channels, input_dim)
        self.pos_embed = PositionalEmbedding((image_size // patch_size) ** 2, input_dim)
        self.encoder = VisionMambaEncoder(input_dim, state_dim, embed_dim, depth, kernel_size, ssm_type)
        self.head = ClassificationHead(input_dim, num_classes)

    def forward(self, x):
        # x: (batch_size, in_channels, image_size, image_size)
        x = self.patch_embed(x) # (batch_size, num_patches, embed_dim)
        x = self.pos_embed(x) # add positional embeddings
        x = self.encoder(x) # process through VSS blocks
        x = self.head(x) # classification output
        return x

## Cifar10

In this note book we will train Vim on Cifar10 for classification task.

In [11]:
# define transformations
transform = transforms.Compose([
    transforms.ToTensor()
])

# load dataset
train_dataset = torchvision.datasets.CIFAR10(
    root='../../../datasets', 
    train=True, 
    download=True, 
    transform=transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root='../../../datasets',
    train=False,
    download=True,
    transform=transform
)

batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## Training Vision Mamba Variants on CIFAR-10

In this section, we train two variants of the Vision Mamba architecture on the CIFAR-10 dataset:

- **Recurrent SSM-based Vision Mamba**  
  This version uses the original recurrent state-space model (SSM) with explicit sequential recurrence.

- **FastSSM-based Vision Mamba**  
  This version replaces the recurrent SSM with a convolutional approximation, enabling faster and more parallelizable training.

Both models share the same training configuration:
- Dataset: CIFAR-10 (10 classes, 32×32 RGB images)
- Optimizer: AdamW
- Loss: Cross Entropy

We will compare their training speed, accuracy, and loss convergence to understand the trade-offs between recurrence and convolutional approximation.

In [12]:
# Train function
def train_model(model, device, num_epochs=10, lr=0.001, weight_decay=0.01):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    model.to(device)
    model.train()
    for epoch in range(num_epochs):
        start_time = time.time()
        batch_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            batch_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
        epoch_loss = batch_loss / len(train_loader)
        epoch_acc = 100 * correct / total
        epoch_time = time.time() - start_time
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.2f}, '
              f'Accuracy: {epoch_acc:.2f}%, Time: {epoch_time:.2f}s')

# Evaluation function
def evaluate_model(model, device):
    model.to(device)
    model.eval()

    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = 100 * correct / total
    print(f'Test Accuracy: {accuracy:.2f}%')

Set Device

In [13]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Training Phase

## Recurrent Vision Mamba

In [14]:
recurrent_vim = VisionMamba(
    image_size=32,
    patch_size=4,
    in_channels=3,
    input_dim=128,
    embed_dim=64,
    state_dim=64,
    ssm_type='recurrent',
    depth=6,
    num_classes=10
)

train_model(recurrent_vim, device, num_epochs=20)

Epoch [1/20], Loss: 2.11, Accuracy: 20.59%, Time: 159.48s
Epoch [2/20], Loss: 1.97, Accuracy: 28.37%, Time: 182.92s
Epoch [3/20], Loss: 1.92, Accuracy: 30.49%, Time: 148.15s
Epoch [4/20], Loss: 1.88, Accuracy: 31.89%, Time: 148.39s
Epoch [5/20], Loss: 1.85, Accuracy: 33.22%, Time: 172.56s
Epoch [6/20], Loss: 1.82, Accuracy: 34.34%, Time: 178.93s
Epoch [7/20], Loss: 1.80, Accuracy: 34.87%, Time: 183.21s
Epoch [8/20], Loss: 1.78, Accuracy: 35.75%, Time: 162.63s
Epoch [9/20], Loss: 1.77, Accuracy: 36.22%, Time: 180.67s
Epoch [10/20], Loss: 1.75, Accuracy: 36.87%, Time: 179.58s
Epoch [11/20], Loss: 1.74, Accuracy: 37.11%, Time: 161.57s
Epoch [12/20], Loss: 1.73, Accuracy: 38.01%, Time: 142.28s
Epoch [13/20], Loss: 1.72, Accuracy: 38.36%, Time: 151.99s
Epoch [14/20], Loss: 1.71, Accuracy: 38.59%, Time: 189.08s
Epoch [15/20], Loss: 1.70, Accuracy: 39.23%, Time: 173.80s
Epoch [16/20], Loss: 1.69, Accuracy: 39.31%, Time: 166.59s
Epoch [17/20], Loss: 1.67, Accuracy: 39.70%, Time: 179.35s
Epoch 

## Convolutional Vision Mamba

In [15]:
convolutional_vim = VisionMamba(
    image_size=32,
    patch_size=4,
    in_channels=3,
    input_dim=128,
    embed_dim=64,
    kernel_size=7,
    ssm_type='convolutional',
    depth=6,
    num_classes=10
)

train_model(convolutional_vim, device, num_epochs=20)

Epoch [1/20], Loss: 2.04, Accuracy: 24.69%, Time: 33.41s
Epoch [2/20], Loss: 1.77, Accuracy: 35.44%, Time: 41.58s
Epoch [3/20], Loss: 1.59, Accuracy: 42.08%, Time: 42.86s
Epoch [4/20], Loss: 1.47, Accuracy: 46.62%, Time: 40.89s
Epoch [5/20], Loss: 1.35, Accuracy: 50.66%, Time: 41.57s
Epoch [6/20], Loss: 1.25, Accuracy: 54.60%, Time: 41.66s
Epoch [7/20], Loss: 1.16, Accuracy: 58.05%, Time: 45.55s
Epoch [8/20], Loss: 1.06, Accuracy: 61.57%, Time: 49.22s
Epoch [9/20], Loss: 0.96, Accuracy: 65.60%, Time: 43.09s
Epoch [10/20], Loss: 0.86, Accuracy: 69.07%, Time: 42.24s
Epoch [11/20], Loss: 0.75, Accuracy: 73.05%, Time: 41.18s
Epoch [12/20], Loss: 0.65, Accuracy: 77.02%, Time: 53.82s
Epoch [13/20], Loss: 0.56, Accuracy: 80.29%, Time: 44.42s
Epoch [14/20], Loss: 0.47, Accuracy: 83.50%, Time: 48.69s
Epoch [15/20], Loss: 0.40, Accuracy: 85.92%, Time: 42.96s
Epoch [16/20], Loss: 0.35, Accuracy: 87.66%, Time: 43.56s
Epoch [17/20], Loss: 0.30, Accuracy: 89.58%, Time: 44.11s
Epoch [18/20], Loss: 0.

# Evaluating Phase

## Recurrent Vision Mamba

In [16]:
evaluate_model(recurrent_vim, device)

Test Accuracy: 37.62%


## Convolutional Vision Mamba

In [17]:
evaluate_model(convolutional_vim, device)

Test Accuracy: 53.40%


### Summary

| Model Type       | Train Accuracy | Test Accuracy | Avg Epoch Time |
|------------------|----------------|----------------|----------------|
| Recurrent SSM    | 41.00%         | 37.62%         | ~170s          |
| FastSSM (Conv)   | 92.52%         | 53.40%         | ~44s           |

To improve performance:
- Increase depth (more blocks)
- Use larger embedding sizes
- Add dropout and weight decay
- Use augmentations (like CutMix, MixUp, RandAugment)
- Train for more epochs